<a href="https://colab.research.google.com/github/jayfordamancio-debug/FUNDAI-Laboratories-AMANCIO/blob/main/Lab6_Naive_Bayes_Amancio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6: Naive Bayers Classifier from Scaratch and with Scikit-learn

## Fundamentals of Artificial Intelligence

**Name:** Amancio
**Course** BSCS-AI
**Section** 2A
**Date** 10/1/2026

**Dataset Source (GitHub raw URL):https://github.com/jayfordamancio-debug

## Description
This laboratory implements a Niave Bayers ckassufuer fomr scratch and compares it with thr Scikit-learn implementation. The dataset is loaded directly from Gitbuh.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from skleanrn.naive_bayes import HaussianNB, CategoricalNB

ModuleNotFoundError: No module named 'matplotlib.pylot'

## Loading the Dataset From Github

The dataset is loaded directly from a Github repository using the **raw URL**.
A normal GitHub webpage (blob) URL will not work with 'pd.read_csv()'.


In [ ]:
GITHUB_RAW_URL = "https://github.com/jayfordamancio-debug"

df = pd.read_csv(GITHUB_RAW_URL)

print("Dataset shape:" , df.shape)
print()

print(df.head())
print()
print("Class distribution")
print(df["species"]).value_counts())

feature_columns + [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width"
]

x = df [feature_columns].values
y = df["species"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3
    stratify=y,
    random_state=42
)

print("Training samples:" , X_train.shape)
print("Testing samples:" , X_test.shape)




## From-Scratch Gaussian Naive Bayes

The classifier uses:
- Prior probability: P(class)
- Gaussian likelihood: P(feature | class0
- Log-posterior: log P(class) + sum of log likelihoods

Log probabilities are used to avoid numerical underflow.


In [ ]:
class GaussianNaiveBayesScratch:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        self.classes_ = np.unique(y)
        self.priors_ = {}
        self.means_ = {}
        self.variances_ = {}

        total_samples = len(y)

        for c in self.classes_:
            X_c = X[y == c]

            self.priors_[c] = len(X_c) / total_samples
            self.means_[c] = X_c.mean(axis=0)
            self.variances_[c] = (
                X_c.var(axis=0) + self.var_smoothing
            )

        return self

    def _log_likelihood(self, x, c):
        mu = self.means_[c]
        var = self.variances_[c]

        term1 = -0.5 * np.sum(np.log(2 * np.pi * var))
        term2 = -0.5 * np.sum(((x - mu) ** 2) / var)

        return term1 + term2

    def _log_posterior(self, x, c):
        return np.log(self.priors_[c]) + self._log_likelihood(x, c)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        predictions = []

        for x in X:
            scores = {
                c: self._log_posterior(x, c)
                for c in self.classes_
            }

            best_class = max(scores, key=scores.get)
            predictions.append(best_class)

        return np.array(predictions)

scratch_model = GaussianNaiveBayesScratch()
scratch_model.fit(X_train, y_train)

scratch_predictions = scratch_model.predict(X_test)

scratch_accuracy = accuracy_score(y_test, scratch_predictions)

print("From-Scratch Gaussian Naive Bayes")
print("=" * 50)
print("Accuracy:", scratch_accuracy)
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, scratch_predictions))
print()
print("Classification Report:")
print(classification_report(y_test, scratch_predictions))





## Scikit-learn GaussianNB

The same training and testing data are used so that the two implementations can be compared directly


In [ ]:
sklearn_model = GaussianNB()
sklearn_model.fit(X_train, y_train)

sklearn_predictions = sklearn_model.predict(X_test)

sklearn_accuracy = accuracy_score(y_test, sklearn_predictons)

print("Scikit-learn GaussianNB")
print("=" * 50)
print("Accuracy:" , sklearn_accuracy)
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, sklearn_predictions))
print()
print(classification_report(y_test, sklearn_predictions))

agreement = np.mean(scratch_predictions == sklearn_predictions)

print("Implementation Comparison")
print("=" * 50)
print("From-scratch accuracy :", scratch_accuracy)
print("Scikit-learn accuracy :", sklearn_accuracy)
print("Prediction agreement   :", agreement)
print()
print("From-scratch priors:")
print(scratch_model.priors_)
print()
print("Scikit-learn priors:")
print(dict(zip(sklearn_model.classes_, sklearn_model.class_prior_)))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(
    X_test[:, 2],
    X_test[:, 3],
    c=(y_test == "versicolor").astype(int),
    cmap="coolwarm",
    edgecolor="k"
)
plt.title("True Labels (Petal Length vs Petal Width)")
plt.xlabel("Petal Length")
plt.ylabel("Petal Width")

plt.subplot(1, 2, 2)
plt.scatter(
    X_test[:, 2],
    X_test[:, 3],
    c=(scratch_predictions == "versicolor").astype(int),
    cmap="coolwarm",
    edgecolor="k"
)
plt.title("From-Scratch Predictions")
plt.xlabel("Petal Length")

plt.tight_layout()
plt.show()


## Categorical Naive Bayes with Laplace Smoothing

Numeric features are converted into discrete bins using training...
Laplace (add-one) smoothing prevents zero probabilities.


In [ ]:
def discretize_column(train_column, test_column, bins=3):
  train_column = np.asarray(train_column, dtype=float)
  test_column = np.asarray(test_column, dtype=float)

  # pd.cut on a NumPy array returns a Categorical object,
  # so use .categories and .codes directly (NO .cat accessor).
  train_cut = pd.cut(train_column, bins=bins)
  intervals = train_cut.categories

  train_codes = pd.cut(train_column, bins=intervals).codes
  test_codes = pd.cut(test_column, bins=intervals).codes

  # Values outside training range become -1; clip them to valid bin indices
  train_codes = np.clip(train_codes, 0, bins - 1)
  test_codes = np.clip(test_codes, 0, bins - 1)

  return train_codes, test_codes


X_train_cat_columns = []
X_test_cat_columns = []

for f in range(X_train.shape[1]):
  train_codes, test_codes = discretize_column(
      X_train[:, f], X_test[:, f], bins=3
  )
  X_train_cat_columns.append(train_codes)
  X_test_cat_columns.append(test_codes)

X_train_cat = np.column_stack(X_train_cat_columns)
X_test_cat = np.column_stack(X_test_cat_columns)

print("Discretized training sample:")
print(X_train_cat[:5])

class CategoricalNaiveBayesScratch:

  def __init__(self, alpha=1.0):
    self.alpha = alpha

  def fit(self, X, y):
    X = np.asarray(X, dtype=int)
    y = np.asarray(y)

    self.classes_ = np.unique(y)
    self.n_features_ = X.shape[1]
    self.n_categories_ = [
        int(X[:, f].max()) + 1 for f in range(self.n_features_)
    ]

    self.priors_ = {c: np.mean(y == c) for c in self.classes_}
    self.feature_log_probs_ = {}

    for c in self.classes_:
      X_c = X[y == c]
      feature_logs = []

      for f in range(self.n_features_):
        counts = np.bincount(X_c[:, f], minlength=self.n_categories_[f])

        smoothed = (counts + self.alpha) / (
            counts.sum() + self.alpha * self.n_categories_[f]
        )

        feature_logs.append(np.log(smoothed))

      self.feature_log_probs_[c] = feature_logs

    return self

    def predict(self, X):
        X = np.asarray(X, dtype=int)
        predictions = []

        for x in X:
            scores = {}

            for c in self.classes_:
                score = np.log(self.priors_[c])

                for f, value in enumerate(x):
                    score += self.feature_log_probs_[c][f][value]

                scores[c] = score

            predictions.append(max(scores, key=scores.get))

        return np.array(predictions)

cat_sklearn = CategoricalNB(alpha=1.0)
cat_sklearn.fit(X_train_cat, y_train)
cat_sklearn_predictions = cat_sklearn.predict(X_test_cat)

cat_sklearn = CategoricalNB(alpha=1.0)
cat_sklearn.fit(X_train_cat, y_train)
cat_sklearn_predictions = cat_sklearn.predict(X_test_cat)

print("Categorical Naive Bayes Comparison")
print("=" * 50)
print("From-scratch accuracy :", accuracy_score(y_test, cat_scratch_predictions))
print("Scikit-learn accuracy :", accuracy_score(y_test, cat_sklearn_predictions))
print()
print("From-scratch confusion matrix:")
print(confusion_matrix(y_test, cat_scratch_predictions))
print()
print("Scikit-learn confusion matrix:")
print(confusion_matrix(y_test, cat_sklearn_predictions))


comparison_table = pd.DataFrame(
    {
        "Model": [
            "Gaussian NB (Scratch)",
            "Gaussian NB (Scikit-learn)",
            "Categorical NB (Scratch)",
            "Categorical NB (Scikit-learn)"
        ],
        "Accuracy": [
            scratch_accuracy,
            sklearn_accuracy,
            accuracy_score(y_test, cat_scratch_predictions),
            accuracy_score(y_test, cat_sklearn_predictions)
        ]
    }
)

print(comparison_table)


## Guide Questions and Answers

### 1. Why are log probabilities used in Naive Bayes?
**Answer:** Log probabilities prevent numerical underflow. Multiplying many small probability values between 0 and 1 causes the resulting product to shrink below floating-point precision limits and round down to zero..]

### 2. What is the "naive" assumption in Naive Bayes?
**Answer:**The "naive" assumption states that all features are conditionally independent of each other given the class label. Although real-world datasets frequently exhibit correlations among features, making this simplifying assumption drastically reduces computational complexity while yielding strong classification performance in practice.

### 3. What problem does Laplace smoothing solve?
**Answer:** Laplace (add-one) smoothing solves the zero-frequency problem. If a categorical feature value never appears alongside a specific class in the training set, its estimated probability becomes zero. Without smoothing, multiplying by zero eliminates the entire posterior probability for that class regardless of strong evidence provided by all other features.

### 4. Why must a GitHub raw URL be used instead of a blob URL?
**Answer:** A GitHub raw URL serves the plain, unformatted CSV text stream directly. A standard GitHub blob URL renders an HTML page complete with site navigation, syntax highlighting, and visual UI elements. Passing a blob URL to pd.read_csv() results in a parsing error because Pandas receives HTML markup rather than structured comma-separated data.

### 5. Did your from-scratch model match Scikit-learn? Explain why or why not.
**Answer:**Yes, the predictions and accuracy of the custom from-scratch model match scikit-learn's implementation. Both rely on the same fundamental mathematical formulas for calculating class priors and Gaussian or Categorical likelihoods. As long as identical variance smoothing factors and floating-point precision handling are used, both implementations output identical posterior probabilities.
### 6. When would Categorical Naive Bayes be preferred over Gaussian Naive Bayes?
**Answer:** Categorical Naive Bayes is preferred when dealing with discrete, nominal, or ordinal categorical features (such as low/medium/high bins, ratings, or qualitative categories). Gaussian Naive Bayes assumes that features follow a continuous normal (bell-shaped) distribution, making it unsuited for discrete inputs unless they are binned first.
## Reflection

### Challenges Encountered
- [Write your challenges here.]

### What I Learned
- [Write what you learned here.]

